# Checkpoint Three: Cleaning Data

Now you are ready to clean your data. Before starting coding, provide the link to your dataset below.

My dataset:

Import the necessary libraries and create your dataframe(s).

In [74]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

bgg_data = r'C:\Users\willm\Desktop\launchcode\cleaning-data-checkpoint\bggdata2023\bgg_GameItem.csv'
bgg_theme = r'C:\Users\willm\Desktop\launchcode\cleaning-data-checkpoint\bggdata2023\bgg_Category.csv'

def reimport_csv(filepath):
    return pd.read_csv(filepath)

df = reimport_csv(bgg_data)
df_theme = reimport_csv(bgg_theme)

df = df.sort_values('rank', ascending=True)

theme_map = df_theme.set_index('bgg_id')['name'].to_dict()

def map_id_to_theme(theme_ids):
    if pd.isna(theme_ids):
        return theme_ids
    ids = theme_ids.split(',')
    names = [theme_map.get(int(cat_id.strip()), 'Unknown') for cat_id in ids]
    return names


df['category'] = df['category'].apply(map_id_to_theme)
pd.set_option('display.max_columns', None)

In [75]:
df_filtered = df[df['rank'].notna()]

df_filtered_small = df[df['rank'] <= 2000]


In [76]:
df_exploded = df.explode('category')
df_grouped = df_exploded.groupby(['year','category']).size().reset_index(name='count')

df_filtered_exploded = df_filtered.explode('category')
df_filtered_grouped = df_filtered_exploded.groupby(['year','category']).size().reset_index(name='count')

df_fs_exploded = df_filtered_small.explode('category')
df_fs_grouped = df_fs_exploded.groupby(['year','category']).size().reset_index(name='count')

## Missing Data

Test your dataset for missing data and handle it as needed. Make notes in the form of code comments as to your thought process.

In [83]:
# Almost all of BGG's data is user provided. As such, games which have fewer players often have less data.
# If a game does not have many user reviews/ratings, BGG will not rank it. This prevents games which no one
# has played from becoming the last ranked games on the site. Therefore there are MANY games with a rank of
# NaN. Similarly, because people have not submitted information, there will be NaN values for such columns
# as 'min_players_rec' or 'complexity'. I can't replace that information, because I don't have the information.
# Nor can I drop it, because it doesn't mean the rest of the data is invalid.

print(df.isna().sum())

print(df_filtered.isna().sum())

print(df_filtered_small.isna().sum())



bgg_id                      0
name                        0
year                     9685
game_type               88074
designer                18251
artist                  66032
publisher                  52
min_players              1938
max_players              5456
min_players_rec          1938
max_players_rec          5456
min_players_best         1938
max_players_best         5456
min_age                 23263
min_age_rec            112778
min_time                22562
max_time                22562
category                 2214
mechanic                16767
cooperative            107550
compilation            112989
compilation_of         112989
family                  34859
implementation         108039
integration            109784
rank                    89425
num_votes               27722
avg_rating              27722
stddev_rating           45407
bayes_rating            88970
complexity              65614
language_dependency    113050
bga_id                 113904
dbpedia_id

## Irregular Data

Detect outliers in your dataset and handle them as needed. Use code comments to make notes about your thought process.

In [96]:
pd.set_option('display.max_rows', None)
df_fs_duplicates = df_filtered_small[(df_filtered_small['rank'].duplicated('keep'==False) == True)][['bgg_id', 'name', 'rank']]
display(df_fs_duplicates)

,bgg_id,name,rank
61160,155821,Inis,100.0
65399,172386,Mombasa,100.0
63210,163412,Patchwork,110.0
58139,144733,Russian Railroads,110.0
47693,102680,Trajan,110.0
8369,9609,War of the Ring,165.0
93337,304783,Hadrian's Wall,165.0
68344,182631,Star Realms: Colony Wars,198.0
18347,21050,Combat Commander: Europe,198.0
88308,283155,Calico,199.0


## Unnecessary Data

Look for the different types of unnecessary data in your dataset and address it as needed. Make sure to use code comments to illustrate your thought process.

In [64]:
# I will drop those columns which will be fairly irrelevant to the particular analysis I want to perform.
# It is possible this data would be useful for further analysis of other questions.
# For example, what themes are most popular for solo games?

df.drop([
    'min_players',
    'max_players',
    'min_players_rec',
    'max_players_rec',
    'min_players_best',
    'max_players_best',
    'bga_id', 
    'dbpedia_id', 
    'luding_id', 
    'spielen_id', 
    'wikidata_id', 
    'wikipedia_id'], axis=1, inplace=True)

display(df)

,bgg_id,name,year,game_type,designer,artist,publisher,min_age,min_age_rec,min_time,max_time,category,mechanic,cooperative,compilation,compilation_of,family,implementation,integration,rank,num_votes,avg_rating,stddev_rating,bayes_rating,complexity,language_dependency
76224,224517,Brass: Birmingham,2018.0,5497,"32887,32943,6","70571,70568,38179","21765,3475,25074,21608,11043,34522,31071,8832,...",14.0,13.825758,60.0,120.0,"[Economic, Industry / Manufacturing, Post-Napo...","2040,2902,2904,2900,2081,3100,2849,2826,2897",NaN,NaN,NaN,"17519,65191,14759,8374,22135,77349,70948,26397...",28720,NaN,1.0,38651.0,8.61232,1.42368,8.42330,3.8955,1.023810
62813,161936,Pandemic Legacy: Season 1,2015.0,"5496,5497","442,378",14057,"538,15889,2366,5657,8820,1391,15983,8291,5812,...",13.0,11.760000,60.0,60.0,"[Environmental, Medical]","2001,2023,2040,2824,2078,2822,2004,3100,2008,2015",1.0,NaN,NaN,"64952,65191,3430,24281,25404,61854,72224,78680...",30549,NaN,2.0,50631.0,8.53562,1.60680,8.38972,2.8318,4.085938
66014,174430,Gloomhaven,2017.0,"5496,5497",69802,"77084,78961,84269","27425,4304,46179,3475,22380,15605,40478,8820,1...",14.0,13.602113,60.0,120.0,"[Adventure, Exploration, Fantasy, Fighting, Mi...","2689,2839,2018,2857,2893,2023,2854,3004,2676,2...",1.0,NaN,NaN,"59218,25158,65191,66335,68438,7005,5615,8374,7...",NaN,"295770,291457",3.0,58418.0,8.62311,1.75165,8.38865,3.8959,4.166667
102831,342942,Ark Nova,2021.0,5497,138517,"138547,11462,12484,11797","22380,30958,21608,10768,12540,8820,42325,17179...",14.0,12.442623,90.0,150.0,"[Animals, Economic, Environmental]","2875,2040,2026,2902,2914,2041,2004,2819,3100,2...",NaN,NaN,NaN,"67874,70360,73596,76649,5666,68335,66167,76846",NaN,NaN,4.0,31052.0,8.53405,1.38518,8.30236,3.7249,3.712121
78094,233078,Twilight Imperium: Fourth Edition,2017.0,"5496,5497","96049,6651,21",11988,"17,23043,3475,157,15889,2973,4617,15605,18852,...",14.0,14.135135,240.0,480.0,"[Civilization, Economic, Exploration, Negotiat...","2838,2080,2021,2072,2843,2676,2026,2914,2886,2...",NaN,NaN,NaN,"67874,64949,25158,29,12210,78680",12493,NaN,5.0,20922.0,8.61973,1.61439,8.24243,4.3080,4.093750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113899,391565,Champions!,2023.0,NaN,"62310,62311,6736",NaN,"4384,1391",10.0,NaN,30.0,45.0,[Party Game],2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113900,391715,Leitin að stjörnunni,2022.0,NaN,"154290,123339,154289",NaN,54258,5.0,NaN,30.0,30.0,"[Children's Game, Movies / TV / Radio theme, P...","2073,2038,2019",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113901,391720,The String Railway Collection,2024.0,NaN,39436,113637,44392,10.0,NaN,20.0,45.0,"[Territory Building, Trains]","2011,2081,2007",NaN,1.0,"76674,100473",11331,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113902,391834,SpellBook,2023.0,NaN,8347,45292,25842,12.0,NaN,45.0,45.0,[Fantasy],"2040,2004,2819",NaN,NaN,NaN,22184,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [67]:
# Here I am re-defining the filtered, small filtered, and exploded datasets with the pared down beginning dataframe.
df_filtered = df[df['rank'].notna()]

df_filtered_small = df[df['rank'] <= 2000]

df_exploded = df.explode('category')
df_grouped = df_exploded.groupby(['year','category']).size().reset_index(name='count')

df_filtered_exploded = df_filtered.explode('category')
df_filtered_grouped = df_filtered_exploded.groupby(['year','category']).size().reset_index(name='count')

df_fs_exploded = df_filtered_small.explode('category')
df_fs_grouped = df_fs_exploded.groupby(['year','category']).size().reset_index(name='count')

In [71]:
# I take the 'exploded' dataframe and look for unique values in the 'category' column.
df_exploded['category'].unique()

array(['Economic', 'Industry / Manufacturing', 'Post-Napoleonic',
       'Trains', 'Transportation', 'Environmental', 'Medical',
       'Adventure', 'Exploration', 'Fantasy', 'Fighting', 'Miniatures',
       'Animals', 'Civilization', 'Negotiation', 'Political',
       'Science Fiction', 'Space Exploration', 'Wargame',
       'Territory Building', 'Movies / TV / Radio theme', 'Novel-based',
       'Civil War', 'Mythology', 'Card Game', 'Modern Warfare',
       'American West', 'Dice', 'Medieval', 'Ancient', 'City Building',
       'Horror', 'Nautical', 'Farming', 'Puzzle', 'Educational',
       'Collectible Components', 'Travel', 'Religious',
       'Comic Book / Strip', 'Spies/Secret Agents', 'Murder/Mystery',
       'Pirates', 'Action / Dexterity', 'Video Game Theme',
       'Mature / Adult', 'Bluffing', 'Abstract Strategy', 'Renaissance',
       'Arabian', 'Racing', 'Sports', 'Prehistoric', 'Deduction',
       'Party Game', 'Word Game', 'Aviation / Flight', 'Number',
       'World W

In [72]:
# Here I am removing the 'categories' which I would define as 'mechanics' rather than 'themes'. 
# Unfortunately this is more an opinion than a factual definition,
# but I've done the best I can to select only those categories which I believe are distinct themes.


df_theme_exploded = df_exploded[
    (df_exploded['category'] != 'Card Game') |
    (df_exploded['category'] != 'Bluffing') |
    (df_exploded['category'] != 'Print & Play') |
    (df_exploded['category'] != 'Real-time') |
    (df_exploded['category'] != 'Action / Dexterity') |
    (df_exploded['category'] != 'Dice') |
    (df_exploded['category'] != 'Collectible Components')]
df_theme_filtered_exploded = df_filtered_exploded[
    (df_filtered_exploded['category'] != 'Card Game') |
    (df_filtered_exploded['category'] != 'Bluffing') |
    (df_filtered_exploded['category'] != 'Print & Play') |
    (df_filtered_exploded['category'] != 'Real-time') |
    (df_filtered_exploded['category'] != 'Action / Dexterity') |
    (df_filtered_exploded['category'] != 'Dice') |
    (df_filtered_exploded['category'] != 'Collectible Components')]
df_theme_fs_exploded = df_fs_exploded[
    (df_fs_exploded['category'] != 'Card Game') |
    (df_fs_exploded['category'] != 'Bluffing') |
    (df_fs_exploded['category'] != 'Print & Play') |
    (df_fs_exploded['category'] != 'Real-time') |
    (df_fs_exploded['category'] != 'Action / Dexterity') |
    (df_fs_exploded['category'] != 'Dice') |
    (df_fs_exploded['category'] != 'Collectible Components')]

## Inconsistent Data

Check for inconsistent data and address any that arises. As always, use code comments to illustrate your thought process.

## Summarize Your Results

Make note of your answers to the following questions.

1. Did you find all four types of dirty data in your dataset?
2. Did the process of cleaning your data give you new insights into your dataset?
3. Is there anything you would like to make note of when it comes to manipulating the data and making visualizations?